# CALM — Adaptive Multimodal Wildfire Monitoring

Demo hệ thống AI giám sát cháy rừng: Planning → Routing → Execution → Evaluation.

- **Plan-driven:** PlanningAgent → RouterAgent → ExecutionAgent
- **Agents:** DataKnowledge, Prediction (SeasFire/heuristic), QA, RSEN, Evaluator

## 1. Setup

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path("/home/hungvu/code/khanh/v2").resolve()
CALM_DIR = ROOT / "calm-"
SEASFIRE_DIR = ROOT / "seasfire-ml"
DATACUBE_DIR = ROOT / "seasfire-datacube"
DATA_DIR = ROOT / "data"

# Nếu file .env nằm trong calm-
ENV_PATH = CALM_DIR / ".env"


def load_env_file(env_path: Path):
    if not env_path.exists():
        print(f"Không thấy file .env tại {env_path}")
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        if line.startswith("export "):
            line = line[len("export "):]

        if "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()

        if (
            (value.startswith('"') and value.endswith('"'))
            or (value.startswith("'") and value.endswith("'"))
        ):
            value = value[1:-1]

        os.environ[key] = value

    print(f"Đã load .env thủ công từ: {env_path}")


print("ROOT =", ROOT)
print("CALM_DIR =", CALM_DIR)
print("ENV_PATH =", ENV_PATH)
print("Python =", sys.version)

# kiểm tra repo CALM
if not (CALM_DIR / "pyproject.toml").exists():
    raise FileNotFoundError(f"Không thấy pyproject.toml trong {CALM_DIR}")

# kiểm tra các repo cần có sẵn
if not SEASFIRE_DIR.exists():
    raise FileNotFoundError(
        f"Không thấy thư mục seasfire-ml tại {SEASFIRE_DIR}. "
        "Script này đã tắt clone nên repo phải có sẵn."
    )

if not DATACUBE_DIR.exists():
    raise FileNotFoundError(
        f"Không thấy thư mục seasfire-datacube tại {DATACUBE_DIR}. "
        "Script này đã tắt clone nên repo phải có sẵn."
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)

# add src vào sys.path trước khi import calm
CALM_SRC = CALM_DIR / "src"
if CALM_SRC.exists() and str(CALM_SRC) not in sys.path:
    sys.path.insert(0, str(CALM_SRC))

# nếu cần import code từ seasfire repos
for p in [SEASFIRE_DIR, DATACUBE_DIR]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# load .env mà không cài thêm thư viện
load_env_file(ENV_PATH)

# set mặc định nếu .env chưa có
os.environ.setdefault("SEASFIRE_ML_PATH", str(SEASFIRE_DIR))
os.environ.setdefault("SEASFIRE_DATASET_PATH", str(DATA_DIR))
os.environ.setdefault("SEASFIRE_DATACUBE_PATH", str(DATACUBE_DIR))
os.environ.setdefault(
    "SEASFIRE_CHECKPOINT",
    str(SEASFIRE_DIR / "runs" / "gru_target-4_oci-0_time-l6.best_model.pt")
)

# in ra các biến quan trọng đã load
keys_to_show = [
    "OPENROUTER_API_KEY",
    "OPENAI_API_KEY",
    "ENABLE_GEE",
    "ENABLE_COPERNICUS",
    "ENABLE_WEB_SEARCH",
    "ENABLE_ARXIV",
    "GEE_PROJECT_ID",
    "COPERNICUS_API_KEY",
    "COPERNICUS_URL",
    "MAX_NEWS_RESULTS",
    "MAX_ARXIV_PAPERS",
    "DEDUP_CHECK",
    "SEASFIRE_ML_PATH",
    "SEASFIRE_DATASET_PATH",
    "SEASFIRE_DATACUBE_PATH",
    "SEASFIRE_CHECKPOINT",
]

print("\n===== ENV CHECK =====")
for k in keys_to_show:
    v = os.environ.get(k)
    if v is None:
        print(f"{k} = <missing>")
    elif "KEY" in k or "TOKEN" in k or "SECRET" in k:
        masked = v[:6] + "..." + v[-4:] if len(v) > 12 else "***"
        print(f"{k} = {masked}")
    else:
        print(f"{k} = {v}")

print("\nSetup OK")

ROOT = /home/hungvu/code/khanh/v2
CALM_DIR = /home/hungvu/code/khanh/v2/calm-
ENV_PATH = /home/hungvu/code/khanh/v2/calm-/.env
Python = 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
Đã load .env thủ công từ: /home/hungvu/code/khanh/v2/calm-/.env

===== ENV CHECK =====
OPENROUTER_API_KEY = sk-or-...8b1a
OPENAI_API_KEY = <missing>
ENABLE_GEE = true
ENABLE_COPERNICUS = true
ENABLE_WEB_SEARCH = true
ENABLE_ARXIV = true
GEE_PROJECT_ID = utilitarian-bee-490908-r7
COPERNICUS_API_KEY = 5ac6ef...fdcd
COPERNICUS_URL = https://cds.climate.copernicus.eu/api
MAX_NEWS_RESULTS = 3
MAX_ARXIV_PAPERS = 3
DEDUP_CHECK = false
SEASFIRE_ML_PATH = /home/hungvu/code/khanh/v2/seasfire-ml
SEASFIRE_DATASET_PATH = /home/hungvu/code/khanh/v2/data
SEASFIRE_DATACUBE_PATH = /home/hungvu/code/khanh/v2/seasfire-datacube
SEASFIRE_CHECKPOINT = home/hungvu/code/khanh/v2/seasfire-ml/runs/gru_target-4_oci-0_time-l6.best_model.pt

Setup OK


In [2]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    max_completion_tokens=1000,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1",
)

/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Planning Agent

URSA 3-node: Generator → Reflector → Formalizer. Output: plan JSON với step_id, action, agent, prompt.

In [3]:
from calm.agents.planning_agent import PlanningAgent

planner = PlanningAgent(llm=llm, config={}, n_max=3, f_max=3)
query = "Wildfire risk assessment for Amazon region next 7 days"
result = planner.invoke(query)

plan = result.get("final_output") or []
print("Approved:", result.get("approved"))
print("Steps:", len(plan))
for i, s in enumerate(plan, 1):
    print(f"  {i}. {s.get('step_id')} | {s.get('action')} | {s.get('agent')}")
    if s.get("prompt"):
        print(f"     prompt: {s['prompt'][:80]}...")

Approved: True
Steps: 7
  1. step-1 | web_search_and_retrieve_knowledge | Environmental Monitor Agent
     prompt: Gather current meteorological data and assess vegetation types in the Amazon reg...
  2. step-2 | retrieve_knowledge | Environmental Monitor Agent
     prompt: Obtain historical wildfire data for the Amazon region....
  3. step-3 | web_search | Environmental Monitor Agent
     prompt: Obtain satellite imagery data for the Amazon region....
  4. step-4 | validate_data | Environmental Monitor Agent
     prompt: Cross-reference and validate the gathered meteorological, vegetation, and satell...
  5. step-5 | predict | Predictive Analysis Agent
     prompt: Analyze collected and validated data to forecast wildfire risk for the next 7 da...
  6. step-6 | compile_report | Reporting Agent
     prompt: Compile all findings into a comprehensive risk assessment report and prepare to ...
  7. step-7 | notify_stakeholders | Reporting Agent
     prompt: Inform local emergency managemen

## 3. Router Agent

Xác định task_type (qa | prediction | hybrid) từ plan + query.

In [4]:
from calm.agents.router_agent import RouterAgent

router = RouterAgent(llm=llm, config={})
routing = router.route(query, plan)
print("Task type:", routing.task_type)
print("Confidence:", routing.confidence)
print("Required artifacts:", routing.required_artifacts)

Task type: prediction
Confidence: 0.9
Required artifacts: ['prediction', 'met_data', 'spatial_data']


## 4. DataKnowledgeAgent

Collect (GEE, CDS, Web, ArXiv) → Extract → Retrieve (ChromaDB). GeocodingTool chuyển địa chỉ văn bản → lat, lon.

In [5]:
from calm.agents.data_knowledge_agent import DataKnowledgeAgent
from calm.agents.memory_agent import MemoryAgent
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool
from calm.tools.geocoding import GeocodingTool

safety = SafetyChecker(llm=llm)
web = WebSearchTool(safety_checker=safety, config={"max_news_results": 5})
geocoding = GeocodingTool(safety_checker=safety, config={})
tools = {"earth_engine": None, "copernicus": None, "web_search": web, "arxiv": None, "geocoding": geocoding}

memory = MemoryAgent(long_term_store=ChromaMemoryStore(collection_name="calm_demo", persist_directory=".chroma", k=3))
data_agent = DataKnowledgeAgent(llm=llm, tools=tools, memory_store=memory, config={"dedup_check": False})

params = {"location": "California, USA", "time_range": {"start": "2024-01-01", "end": "2024-12-31"}}
ret = data_agent.retrieve("wildfire causes in California 2024", parameters=params)
print("Retrieved:", len(ret.get("retrieved_data", [])), "sources")
print("Dedup hit:", ret.get("dedup", False))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11224.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieved: 10 sources
Dedup hit: False


## 5. RSEN — Validation

Weather Analyst + Geo Analyst (song song) → Ops Coordinator → Plausible/Implausible.

In [6]:
from calm.agents.rsen_module import RSENModule

rsen = RSENModule(llm=llm, memory_store=memory, k=3)
val = rsen.validate(
    prediction={"risk_level": "High", "confidence": 0.8},
    met_data={"temperature": 35.0, "humidity": 0.2},
    spatial_data={"fuel_type": "Shrubland", "slope": 25},
)
print("Decision:", val.get("validation_decision"))
print("Rationale:", (val.get("final_rationale") or "")[:200]+"...")

Decision: Plausible
Rationale: The initial prediction of high risk is validated by the weather conditions, which promote fire ignition and spread, and the geospatial analysis, which identifies highly flammable shrubland on steep sl...


## 6. Wildfire QA Agent

Evidence Evaluator → retrieve → trả lời với citations.

In [7]:
from calm.agents.qa_agent import WildfireQAAgent

qa = WildfireQAAgent(llm=llm, data_agent=data_agent, web_search_tool=web, memory_store=memory, config={"evidence_threshold": 0.65})
qa_result = qa.invoke("What caused the 2023 Canadian wildfires?")
out = qa_result.get("final_output") or {}
print("Answer:", (out.get("answer", ""))[:300])
print("Citations:", out.get("citations", [])[:2])
print("Confidence:", out.get("confidence"))

Answer: The 2023 Canadian wildfires were primarily driven by a combination of extreme weather conditions, human activities, and the broader impacts of climate change. Key factors included: 

1. **Extreme Weather Conditions**: The summer of 2023 saw unprecedented high temperatures, with the mean temperature 
Citations: ["Natural Resources Canada: Canada's record-breaking wildfires in 2023", 'Biology Insights: 2023 Canadian Wildfires: Causes, Scale, and Consequences']
Confidence: 0.95


## 7. Full Pipeline — CALMOrchestrator

Một lệnh `run(query)` → Planning → Routing → Execution. Tự route QA hoặc Prediction.

In [9]:
import os
import sys
from pathlib import Path

from calm.orchestrator import CALMOrchestrator
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.web_search import WebSearchTool
from calm.tools.safety_check import SafetyChecker
from calm.tools.wildfire_models import WildfireModelRunner

# =========================
# 1) Load env do file .sh tạo ra
# =========================
def load_shell_env(env_file: Path):
    if not env_file.exists():
        raise FileNotFoundError(f"Không thấy file env: {env_file}")

    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):]
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ[key] = value


# =========================
# 2) Chuẩn hóa path từ env
# =========================
def normalize_path(raw_value: str, base_dir: Path | None = None) -> Path:
    """
    Chuẩn hóa path lấy từ env:
    - expand ~ và biến môi trường
    - sửa trường hợp thiếu dấu / đầu -> home/... => /home/...
    - sửa trường hợp bị dính prefix kiểu:
      /home/.../calm-/home/...  -> /home/...
    - nếu là relative path thì ghép với base_dir
    """
    if not raw_value:
        raise ValueError("Path rỗng")

    s = os.path.expandvars(os.path.expanduser(str(raw_value).strip()))

    # Trường hợp bị thiếu dấu / đầu: home/hungvu/... -> /home/hungvu/...
    if s.startswith("home/"):
        s = "/" + s

    # Trường hợp path bị dính prefix, ví dụ:
    # /home/hungvu/code/khanh/v2/calm-/home/hungvu/code/khanh/v2/seasfire-ml/...
    home_idx = s.find("/home/")
    if home_idx > 0:
        s = s[home_idx:]

    p = Path(s)

    # Nếu vẫn là relative path thì nối với base_dir
    if not p.is_absolute():
        if base_dir is None:
            p = p.resolve()
        else:
            p = (base_dir / p).resolve()
    else:
        p = p.resolve()

    return p


# Nếu lúc chạy sh bạn truyền BASE_DIR, sửa đúng chỗ này.
BASE_DIR = Path("/home/hungvu/code/khanh/v2").resolve()

ENV_FILE = BASE_DIR / "seasfire.env"
load_shell_env(ENV_FILE)

SEASFIRE_ML_PATH = normalize_path(os.environ["SEASFIRE_ML_PATH"], BASE_DIR)
SEASFIRE_DATASET_PATH = normalize_path(os.environ["SEASFIRE_DATASET_PATH"], BASE_DIR)
SEASFIRE_DATACUBE_PATH = (BASE_DIR / "seasfire-datacube").resolve()

print("SEASFIRE_ML_PATH      =", SEASFIRE_ML_PATH)
print("SEASFIRE_DATASET_PATH =", SEASFIRE_DATASET_PATH)
print("SEASFIRE_DATACUBE_PATH=", SEASFIRE_DATACUBE_PATH)

# Nếu cần import code từ 2 repo đã clone
for p in [SEASFIRE_ML_PATH, SEASFIRE_DATACUBE_PATH]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))


# =========================
# 3) Tự dò checkpoint đã có sẵn
# =========================
def guess_checkpoint(root: Path):
    if not root.exists():
        return None

    ignore_dirs = {".git", "__pycache__", "venv", ".venv", "node_modules"}
    patterns = ["**/*.ckpt", "**/*.pt", "**/*.pth"]

    candidates = []
    for pattern in patterns:
        for p in root.glob(pattern):
            if any(part in ignore_dirs for part in p.parts):
                continue
            name = p.name.lower()
            if any(k in name for k in ["best", "last", "checkpoint", "ckpt", "model", "firecast"]):
                candidates.append(p)

    if not candidates:
        return None

    candidates.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0]


# Ưu tiên checkpoint từ env, nhưng chỉ dùng nếu file tồn tại
checkpoint = None
env_checkpoint = os.environ.get("SEASFIRE_CHECKPOINT", "").strip()

if env_checkpoint:
    try:
        candidate = normalize_path(env_checkpoint, BASE_DIR)
        if candidate.exists():
            checkpoint = str(candidate)
        else:
            print(f"[WARN] Checkpoint trong env không tồn tại: {candidate}")
    except Exception as e:
        print(f"[WARN] Không parse được SEASFIRE_CHECKPOINT: {e}")

# Nếu checkpoint từ env sai thì tự dò lại trong repo seasfire-ml
if checkpoint is None:
    ckpt = guess_checkpoint(SEASFIRE_ML_PATH)
    checkpoint = str(ckpt.resolve()) if ckpt else None

print("CHECKPOINT =", checkpoint)


# =========================
# 4) Tạo tools + orchestrator
# =========================
memory_store = ChromaMemoryStore(
    collection_name="calm_orch",
    persist_directory=".chroma"
)

tools = {
    "web_search": WebSearchTool(
        safety_checker=SafetyChecker(llm=llm),
        config={"max_news_results": 3},
    )
}

orch_kwargs = dict(
    llm=llm,
    memory_store=memory_store,
    tools=tools,
    config={"planner_n_max": 2},
)

if checkpoint:
    model_runner = WildfireModelRunner({
        "model": os.environ.get("SEASFIRE_MODEL", "firecastnet"),
        "checkpoint": checkpoint,
        "device": os.environ.get("SEASFIRE_DEVICE", "cpu"),
        "dataset_path": str(SEASFIRE_DATASET_PATH),
    })
    model_runner.load()
    orch_kwargs["model_runner"] = model_runner
else:
    print("[WARN] Không tìm thấy checkpoint trong seasfire-ml.")
    print("[WARN] Prediction branch có thể vẫn trả về: model unavailable")

orch = CALMOrchestrator.from_llm(**orch_kwargs)


# =========================
# 5) Chạy thử
# =========================
result = orch.run("Predict wildfire risk California next 7 days")

print("Task type:", result.get("task_type"))
if result.get("task_type") == "qa":
    print("Answer:", (result.get("answer", "") or "")[:250])
else:
    print(
        "Risk:",
        result.get("risk_level"),
        "|",
        result.get("decision"),
        "|",
        (result.get("rationale", "") or "")[:150],
    )

if result.get("error"):
    print("Error:", str(result["error"])[:200])

SEASFIRE_ML_PATH      = /home/hungvu/code/khanh/v2/seasfire-ml
SEASFIRE_DATASET_PATH = /home/hungvu/code/khanh/v2/data/seasfire_v0.4
SEASFIRE_DATACUBE_PATH= /home/hungvu/code/khanh/v2/seasfire-datacube
CHECKPOINT = /home/hungvu/code/khanh/v2/seasfire-ml/runs/gru_target-4_oci-0_time-l6.best_model.pt


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11304.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Task type: prediction
Risk: Unknown | Unknown | 
Error: model unavailable


## 8. Evaluator — LLM-as-a-Judge

Chấm 5 tiêu chí (0–100): Data Accuracy, Explainability, Jargon Avoidance, Redundancy Avoidance, Citation Quality. Passing: average >= 75.

In [ ]:
from calm.agents.evaluator_agent import EvaluatorAgent

evaluator = EvaluatorAgent(llm=llm, config={"passing_score": 75.0})
eval_query = "What causes wildfires in the Amazon?"
orch_result = orch.run(eval_query)  
eval_result = evaluator.evaluate(response=orch_result, query=eval_query)

print("Average score:", eval_result.get("average_score"))
print("Passed:", eval_result.get("passed"))
print("Scores:", eval_result.get("scores", {}))

Average score: 86.0
Passed: True
Scores: {'data_accuracy': 90, 'explainability': 85, 'jargon_avoidance': 90, 'redundancy_avoidance': 95, 'citation_quality': 90}
